# Part 1. Equation of a Slime

How many late days are you using for this assignment?

In [1]:
# Imports section
# NO late day

## 1. Loading the dataset

In [5]:
# Using pandas load the dataset
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)
import pandas as pd

# Read the CSV file
df = pd.read_csv("science_data_large.csv")

# Output the first 15 rows
print("First 15 rows of the dataset:")
print(df.head(15))

# Display a summary of the table information
print("\nSummary of the dataset:")
print(df.info())


First 15 rows of the dataset:
    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04

Summary of the dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL    

## 2. Splitting the dataset

In [6]:
# Take the pandas dataset and split it into our features (X) and label (y)

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
# For grading consistency use random_state=42 
from sklearn.model_selection import train_test_split

# Split into features (X) and label (y)
X = df[["Temperature °C", "Mols KCL"]]
y = df["Size nm^3"]

# Use sklearn to split into training and test sets (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Optional: Print the shapes of the splits to confirm
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (900, 2)
X_test shape: (100, 2)
y_train shape: (900,)
y_test shape: (100,)


## 3. Perform a Linear Regression

In [7]:
# Use sklearn to train a model on the training set

# Create a sample datapoint and predict the output of that sample with the trained model

# Report the score for that model using the default score function property of the SKLearn model, in your own words (markdown, not code) explain what the score means

# Extract the coefficients and intercept from the model and write an equation for your h(x) using LaTeX
# Import libraries
from sklearn.linear_model import LinearRegression
from IPython.display import display, Math

# Train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Create a sample input and predict
sample = pd.DataFrame([[50, 25]], columns=["Temperature °C", "Mols KCL"])
prediction = model.predict(sample)
print(f"Predicted slime size for Temperature = 50°C and Mols KCL = 25: {prediction[0]:.5f} nm^3")

# Score (R²)
score = model.score(X_test, y_test)
print(f"\nModel Score (R²): {score:.4f}")

# Clean values for display
def clean(value):
    return 0 if abs(value) < 1e-5 else round(value, 5)

# Extract and clean coefficients
coef_temp = clean(model.coef_[0])
coef_kcl = clean(model.coef_[1])
intercept = clean(model.intercept_)

# Display the linear equation in LaTeX
equation = f"h(x) = {coef_temp} \\cdot \\text{{Temp}} + {coef_kcl} \\cdot \\text{{KCl}} + ({intercept})"
display(Math(equation))



Predicted slime size for Temperature = 50°C and Mols KCL = 25: -340266.78225 nm^3

Model Score (R²): 0.8552


<IPython.core.display.Math object>

Write the linear equation of a slime: (example equation: $E = mc^2$)

h(x)=866.14641⋅Temp+1032.69507⋅KCl+(−409391.47958)


Report on score and explain meaning: R^2 = 0.8552 represents how well the linear regression model fits the data. The model explains 85.52% of the variation in the slime size based on input temperature and KCI concentration. This means that the majority of the changes in slime size can be predicted accurately using this model. The remaining 14.48% of the variation is due to factors not captured by the model.

## 4. Use Cross Validation

In [8]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
# For grading consistency use n_splits=5 and random_state=42

# Report on their finding and their significance
from sklearn.model_selection import cross_val_score, KFold
import numpy as np

# Set up cross-validation strategy with 5 splits and random_state=42
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Perform cross-validation using R² as the default scoring metric
cv_scores = cross_val_score(model, X, y, cv=cv)

# Print all fold scores
print("Cross-validation scores (R² for each fold):")
print(cv_scores)

# Report mean and standard deviation
mean_score = np.mean(cv_scores)
std_dev = np.std(cv_scores)

print(f"\nMean R² Score: {mean_score:.4f}")
print(f"Standard Deviation: {std_dev:.4f}")


Cross-validation scores (R² for each fold):
[0.86151889 0.82742341 0.87195173 0.88166206 0.85609101]

Mean R² Score: 0.8597
Standard Deviation: 0.0184


Write findings here: These results (86%, 83%, 87%, 88%, and 86%) indicate that the model is consistently performing well across different subsets of the data, with only slight variation in performance. The low standard deviation (0.0184) suggests the model is stable and generalizes well to unseen data.


## 5. Using Polynomial Regression

In [9]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
# Perform k-fold cross validation (as above)

# Report on the metrics and output the resultant equation as you did in Part 3.

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, KFold
import numpy as np
from IPython.display import display, Math

# Create polynomial features of degree 2
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

# Create and train a model pipeline
poly_model = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression())
poly_model.fit(X, y)

# 5-fold cross-validation with shuffling
cv = KFold(n_splits=5, shuffle=True, random_state=42)
poly_scores = cross_val_score(poly_model, X, y, cv=cv)

# Print scores
print("Polynomial Regression Cross-validation scores (R²):")
print(poly_scores)
print(f"\nMean R² Score: {np.mean(poly_scores):.4f}")
print(f"Standard Deviation: {np.std(poly_scores):.4f}")

# Get coefficients from trained model
# We fit again separately so we can extract the actual coefficients (not through the pipeline)
poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly_features.fit_transform(X)
linreg_poly = LinearRegression().fit(X_poly, y)

# Round small values (as done before)
def clean(value):
    return 0 if abs(value) < 1e-5 else round(value, 5)

coefs = [clean(c) for c in linreg_poly.coef_]
intercept = clean(linreg_poly.intercept_)

# Get feature names
feature_names = poly_features.get_feature_names_out(["Temperature °C", "Mols KCL"])

# Build equation in LaTeX
terms = [f"{coef} \\cdot {name}" for coef, name in zip(coefs, feature_names) if coef != 0]
latex_equation = " + ".join(terms) + f" + {intercept}"
display(Math("h(x) = " + latex_equation))

Polynomial Regression Cross-validation scores (R²):
[1. 1. 1. 1. 1.]

Mean R² Score: 1.0000
Standard Deviation: 0.0000


<IPython.core.display.Math object>

Write the polynomial equation of a slime: (example equation: $E = mc^2$)

Report on the score and interpret: The polynomial regression model with degree 2 achieved a perfect score (R^2=1.0) across all 5 cross-validation folds, with zero variation between them. This means the model explains 100% of the variance in the target variable. In other words, the performance is perfectly consistent cross different subsets of the data. A perfect score may also indicate overfitting, especially if the dataset is small or the features are highly correlated.

# Part 2. Chronic Kidney Disease Prediction via Classification

Create code and markdown cells as needed to perform classification and report on your results

In [10]:
# Load the dataset. Then train and evaluate the classification models.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("ckd_feature_subset.csv")

X = df.drop("Target_ckd", axis=1)
y = df["Target_ckd"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Results and Conclusion for Classification Experiments

In [11]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(),
    "Support Vector Machine": SVC(),
    "Random Forest": RandomForestClassifier()
}

for name, model in models.items():
    if name == "Random Forest":
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    else:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    
    print(f"\n===== {name} =====")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


===== Logistic Regression =====
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95        21
           1       0.90      0.90      0.90        10

    accuracy                           0.94        31
   macro avg       0.93      0.93      0.93        31
weighted avg       0.94      0.94      0.94        31

Confusion Matrix:
[[20  1]
 [ 1  9]]

===== Support Vector Machine =====
Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98        21
           1       1.00      0.90      0.95        10

    accuracy                           0.97        31
   macro avg       0.98      0.95      0.96        31
weighted avg       0.97      0.97      0.97        31

Confusion Matrix:
[[21  0]
 [ 1  9]]

===== Random Forest =====
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        2

**Logistic Regression**
- Accuracy: 94%
- Precision: 0.95/0.90
- Recall: 0.95/0.90
- F1-score: 0.95/0.90
- Conclusion: This model performed well with 94% accuracy, but had slightly lower recall for class 1 (patients with CKD).

**SVM**
- Accuracy: 97%
- Precision: 0.95/1.00
- Recall: 1.00/0.90
- F1-score: 0.98/0.95
- Conclusion: This model improved with 97% accuracy, correctly identifying nearly all patients.

**Random Forest**
- Accuracy: 100%
- Precision: 1.00/1.00
- Recall: 1.00/1.00
- F1-score: 1.00/1.00
- Conclusion: This model achieved perfect scores, meaning it correctly classified every test instance.

## Results and Conclusion for Neural Network Experiments

In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlp = MLPClassifier(hidden_layer_sizes=(16, 8),
                    activation='relu',
                    solver='adam',
                    max_iter=1000,
                    random_state=42)

mlp.fit(X_train_scaled, y_train)

y_pred = mlp.predict(X_test_scaled)

print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.98        21
           1       0.91      1.00      0.95        10

    accuracy                           0.97        31
   macro avg       0.95      0.98      0.96        31
weighted avg       0.97      0.97      0.97        31

Confusion Matrix:
[[20  1]
 [ 0 10]]


**Conclusion**
- The MLPClassifier caught all CKD-positive cases (recall=1.00)
- It misclassified 1 healthy person as CKD-positive
- The model made only 1 error out of 31 predictions (97%)
- Summary: The model learned the patterns effectively, especially for identifying patients with CKD.